### Beyond the Squeaky Wheel: 311 Engagement & Equity Analysis
### Notebook 3: 311 Service Request Keyword Matching

Matches the keyword-classified categories back to each city's original 311 service request records. Takes the classifier output and each city's raw 311 CSV as input, and produces a per-city classified CSV plus a cross-city processing summary.

In [ ]:
# Step 1: Import libraries

import pandas as pd
import os

# to clean and normalize special characters
import unicodedata

In [ ]:
# Step 2: Upload data

MASTER_PATH = 'INSERT FILE PATH: classified master keyword match CSV'
CITY_FOLDER = 'INSERT FOLDER PATH: raw 311 city input files'
OUTPUT_FOLDER = 'INSERT FOLDER PATH: matched output files'

In [ ]:
# Step 3: Data prep

master = pd.read_csv(MASTER_PATH)

# keep only the match key and classification columns
# drops City, Service_Request, Count — keeps Combined_Match + all category columns
classification_cols = [c for c in master.columns 
                       if c not in ['City', 'Service_Request', 'Count']]
master_slim = master[classification_cols].copy()

print(f"Master loaded: {len(master_slim):,} rows, {len(classification_cols)} columns")

# standardise master match key once here rather than inside the loop
master_slim['Combined_Match_key'] = master_slim['Combined_Match'].str.lower().str.strip()

master_slim = master_slim.drop_duplicates(subset=['Combined_Match_key'], keep='first')
print(f"Master after dedup: {len(master_slim):,} rows")
print(f"Duplicates removed: {len(master) - len(master_slim):,}")
print(f"Unique keys: {master_slim['Combined_Match_key'].nunique():,}")

In [ ]:
# Sanity Check: review master match data

print(f"Rows: {master_slim.shape[0]}")
print(f"Columns: {master_slim.shape[1]}")
print("--------------")
print(f"Categories: {master_slim.columns.tolist()}")
print("--------------")
master_slim.sample(3)

In [ ]:
# Step 4: Configure cities

# For each city define:
        #   'file': CSV filename
        #   'city_name': exact string to use in Combined_Match (must match master list)
        #   'col1': first service request column
        #   'col2': second column to concat (None if single-column city)

CITY_CONFIG = {
    'Atlanta': {
        'file': 'Atlanta_clean_22-25_noXCL.csv',
        'city_name': 'Atlanta',
        'col1': 'Short.Description',
        'col2': None},
    'Austin': {
        'file': 'Austin_22-25_noXCL.csv',
        'city_name': 'Austin',
        'col1': 'SR.Description',
        'col2': None},
    'Baltimore': {
        'file': 'Baltimore_22-25_noXCL.csv',
        'city_name': 'Baltimore',
        'col1': 'SRType',
        'col2': None},
    'Boston': {
        'file': 'Boston_22-25_noXCL.csv',
        'city_name': 'Boston',
        'col1': 'reason',
        'col2': 'case_title'},
    'Charlotte': {
        'file': 'Charlotte_22-25_noXCL.csv',
        'city_name': 'Charlotte',
        'col1': 'REQUEST_TYPE',
        'col2': 'TITLE'},
    'Cincinnati': {
        'file': 'Cincinnati_22-25.csv',
        'city_name': 'Cincinnati',
        'col1': 'SR_TYPE',
        'col2': 'SR_TYPE_DESC'},
    'Chicago': {
        'file': 'Chicago_22-25_noXCL.csv',
        'city_name': 'Chicago',
        'col1': 'SR_TYPE',
        'col2': None},
    'Cleveland': {
        'file': 'Cleveland_22-25.csv',
        'city_name': 'Cleveland',
        'col1': 'service_category',
        'col2': 'service_name'},
    'Dallas': {
        'file': 'Dallas_22-25_noXCL.csv',
        'city_name': 'Dallas',
        'col1': 'Service.Request.Type',
        'col2': None},
    'Detroit': {
        'file': 'Detroit_22-25.csv',
        'city_name': 'Detroit',
        'col1': 'Request.Type.Title',
        'col2': None},
    'Indianapolis': {
        'file': 'Indy_22-25_clean.csv',
        'city_name': 'Indianapolis',
        'col1': 'SERVICENAME',
        'col2': 'ACTIVITY'},
    'Kansas_City': {
        'file': 'KansasCity_22-25.csv',
        'city_name': 'Kansas City',
        'col1': 'Issue.Type',
        'col2': 'Issue.Sub.Type'},
    'Los_Angeles': {
        'file': 'LosAngeles_22-25_noXCL_full.csv',
        'city_name': 'Los Angeles',
        'col1': 'RequestType',
        'col2': None}, 
    'Miami': {
        'file': 'Miami_clean_22-25_noXCL.csv',
        'city_name': 'Miami',
        'col1': 'issue_type',
        'col2': None},
    'Milwaukee': {
        'file': 'Milwaukee_22-25.csv',
        'city_name': 'Milwaukee',
        'col1': 'TITLE',
        'col2': None},     
    #'Minneapolis': {               #processed in the 3-column sheet
    #    'file': 'Minneapolis_22-25.csv',
    #    'city_name': 'Minneapolis',
    #    'col1': 'SUBJECTNAME',
    #    'col2': 'REASONNAME',
    #    'col3': 'TYPENAME'},
    #'Nashville': {                 #processed in the 3-column sheet
    #    'file': 'Nashville_22-25.csv',
    #    'city_name': 'Nashville',
    #    'col1': 'Request.Type',
    #    'col2': 'Subrequest.Type',
    #    'col3': 'Additional.Subrequest.Type'},
    'New_Orleans': {
        'file': 'NewOrleans_22-25.csv',
        'city_name': 'New Orleans',
        'col1': 'Request.Type',
        'col2': 'Request.Reason'},
    'New_York': {                  #huge, extends run time
        'file': 'NewYork_22-25_noXCL.csv',
        'city_name': 'New York',
        'col1': 'Problem (formerly Complaint Type)',
        'col2': 'Problem Detail (formerly Descriptor)'},
    'Philadelphia': {
        'file': 'Philadelphia_22-25.csv',
        'city_name': 'Philadelphia',
        'col1': 'subject',
        'col2': 'service_name'},
    'San_Diego': {
        'file': 'SanDiego_22-25_noXCL.csv',
        'city_name': 'San Diego',
        'col1': 'service_name',
        'col2': 'service_name_detail'},
    'San_Francisco': {
        'file': 'SanFrancisco_22-25_noXCL.csv',  
        'city_name': 'San Francisco',
        'col1': 'Category',
        'col2': 'Request.Type'},
    'Seattle': {
        'file': 'Seattle_22-25_noXCL.csv',
        'city_name': 'Seattle',
        'col1': 'Service.Request.Type',
        'col2': None},
    #'St_Louis': {                  #processed in the 3-column sheet
    #    'file': 'StLouis_22-25.csv',
    #    'city_name': 'St Louis',
    #    'col1': 'Group',
    #    'col2': 'Description',
    #    'col3': 'PLAIN_ENGLISH_NAME_FOR_PROBLEMCODE'},
    'Washington_DC': {
        'file': 'WashingtonDC_22-25.csv',
        'city_name': 'Washington DC',
        'col1': 'DESCRIPTION',
        'col2': None},
}

In [ ]:
##### Option to create test run on 1 city to confirm functionality and results
##### Removed for clean final code

In [ ]:
# Step 5: Process each city
    # Takes about 35 minutes
    # Special character corruption is siginificant issue
    # test/assess SR rows for complete correction - will cause MatchKey join failures


summary = []

for city, config in CITY_CONFIG.items():
    print(f"\nProcessing {city}...")
    
    try:
        # load city file
        filepath = os.path.join(CITY_FOLDER, config['file'])
        df = pd.read_csv(filepath, low_memory=False)
        print(f"  Loaded: {len(df):,} rows, {len(df.columns)} columns")
        
        # create City column
        df['City'] = config['city_name']
        
        # build Combined_Match column
        col1 = config['col1']
        if config['col2'] is not None:
            col2 = config['col2']
            df['Combined_Match'] = (config['city_name'] + ' - '
                                    + df[col1].fillna('').astype(str) + ' - '
                                    + df[col2].fillna('').astype(str))
        else:
            df['Combined_Match'] = (config['city_name'] + ' - '
                                    + df[col1].fillna('').astype(str))

        # then clean Combined_Match — same order as classifier notebook
        def normalize_chars(text):
            # normalize unicode to closest ASCII equivalent
            return unicodedata.normalize('NFKD', str(text)).encode('ascii', 'ignore').decode('ascii')
 
        #### Punctuation & spacing fixes
        df['Combined_Match'] = df['Combined_Match'].str.replace('&amp;', 'and', regex=False)            # converts HTML ampersand
        df['Combined_Match'] = df['Combined_Match'].str.replace('&', 'and', regex=False)                # converts &
        df['Combined_Match'] = df['Combined_Match'].str.replace('_', ' ', regex=False)                  # removes _
        df['Combined_Match'] = df['Combined_Match'].str.replace(',', '', regex=False)                   # removes ,
        df['Combined_Match'] = df['Combined_Match'].str.replace('\t', ' ', regex=False)                 # removes tab
 
        #### Standard Unicode special characters
        df['Combined_Match'] = df['Combined_Match'].str.replace('\u2013', '-', regex=False)   # en dash –
        df['Combined_Match'] = df['Combined_Match'].str.replace('\u2014', '-', regex=False)   # em dash —
        df['Combined_Match'] = df['Combined_Match'].str.replace('\u2018', "'", regex=False)   # left single quote '
        df['Combined_Match'] = df['Combined_Match'].str.replace('\u2019', "'", regex=False)   # right single quote '
        df['Combined_Match'] = df['Combined_Match'].str.replace('\u201c', '"', regex=False)   # left double quote "
        df['Combined_Match'] = df['Combined_Match'].str.replace('\u201d', '"', regex=False)   # right double quote "
        df['Combined_Match'] = df['Combined_Match'].str.replace('\u2026', '...', regex=False) # ellipsis …
        df['Combined_Match'] = df['Combined_Match'].str.replace('\u00a0', ' ', regex=False)   # non-breaking space
        df['Combined_Match'] = df['Combined_Match'].str.replace('\u00e9', 'e', regex=False)   # é
        df['Combined_Match'] = df['Combined_Match'].str.replace('\u00e8', 'e', regex=False)   # è
        df['Combined_Match'] = df['Combined_Match'].str.replace('\u00ea', 'e', regex=False)   # ê
        df['Combined_Match'] = df['Combined_Match'].str.replace('\u00e0', 'a', regex=False)   # à
        df['Combined_Match'] = df['Combined_Match'].str.replace('\u00e1', 'a', regex=False)   # á
        df['Combined_Match'] = df['Combined_Match'].str.replace('\u00e2', 'a', regex=False)   # â
        df['Combined_Match'] = df['Combined_Match'].str.replace('\u00f3', 'o', regex=False)   # ó
        df['Combined_Match'] = df['Combined_Match'].str.replace('\u00f2', 'o', regex=False)   # ò
        df['Combined_Match'] = df['Combined_Match'].str.replace('\u00ed', 'i', regex=False)   # í
        df['Combined_Match'] = df['Combined_Match'].str.replace('\u00f1', 'n', regex=False)   # ñ
        df['Combined_Match'] = df['Combined_Match'].str.replace('\u00fc', 'u', regex=False)   # ü
        df['Combined_Match'] = df['Combined_Match'].str.replace('\u00e7', 'c', regex=False)   # ç
        df['Combined_Match'] = df['Combined_Match'].str.replace('\u00ae', '', regex=False)    # ® registered trademark
        df['Combined_Match'] = df['Combined_Match'].str.replace('\u2122', '', regex=False)    # ™ trademark
        df['Combined_Match'] = df['Combined_Match'].str.replace('\u00a1', '', regex=False)    # ¡ inverted exclamation
 
        #### Corrupted Mac encoding sequences
        df['Combined_Match'] = df['Combined_Match'].str.replace('‚äì', '-', regex=False)      # corrupted em dash
        df['Combined_Match'] = df['Combined_Match'].str.replace('‚äî', '-', regex=False)      # corrupted em dash variant
        df['Combined_Match'] = df['Combined_Match'].str.replace('‚äô', "'", regex=False)      # corrupted apostrophe
        df['Combined_Match'] = df['Combined_Match'].str.replace('‚äú', '"', regex=False)      # corrupted open double quote
        df['Combined_Match'] = df['Combined_Match'].str.replace('‚äù', '"', regex=False)      # corrupted close double quote
        df['Combined_Match'] = df['Combined_Match'].str.replace('‚äò', "'", regex=False)      # corrupted open single quote
        df['Combined_Match'] = df['Combined_Match'].str.replace('‚ä¶', '...', regex=False)    # corrupted ellipsis
        df['Combined_Match'] = df['Combined_Match'].str.replace('‚ñ¢', '', regex=False)       # corrupted trademark
        df['Combined_Match'] = df['Combined_Match'].str.replace('¬†', ' ', regex=False)       # corrupted non-breaking space
        df['Combined_Match'] = df['Combined_Match'].str.replace('¬°', '', regex=False)        # corrupted inverted exclamation
        df['Combined_Match'] = df['Combined_Match'].str.replace('√©', 'e', regex=False)       # corrupted é
        df['Combined_Match'] = df['Combined_Match'].str.replace('√≥', 'o', regex=False)       # corrupted ó
        df['Combined_Match'] = df['Combined_Match'].str.replace('√°', 'a', regex=False)       # corrupted á
        df['Combined_Match'] = df['Combined_Match'].str.replace('√≠', 'i', regex=False)       # corrupted í
        df['Combined_Match'] = df['Combined_Match'].str.replace('√±', 'n', regex=False)       # corrupted ñ
 
        #### Final cleanup
        df['Combined_Match'] = df['Combined_Match'].str.replace(r'\s+', ' ', regex=True).str.strip()  # collapse spaces before normalize
        df['Combined_Match'] = df['Combined_Match'].apply(normalize_chars)
        df['Combined_Match'] = df['Combined_Match'].str.replace(r'\s+', ' ', regex=True).str.strip()  # collapse any new spaces after normalize
        df['Combined_Match'] = df['Combined_Match'].str.lower()
 
        # standardise for matching
        df['Combined_Match_key'] = df['Combined_Match'].str.lower().str.strip()
        
        # join classification data
        df_classified = df.merge(
            master_slim.drop(columns=['Combined_Match']),
            on='Combined_Match_key',
            how='left')
        
        # drop the temp key column
        df_classified = df_classified.drop(columns=['Combined_Match_key'])
        
        # export
        output_path = os.path.join(OUTPUT_FOLDER, f"{city}_classified.csv")
        df_classified.to_csv(output_path, index=False)
    
        # summary stats
        matched = df_classified['Any_Match'].sum() if 'Any_Match' in df_classified.columns else 'N/A'
        total = len(df_classified)
        unjoined = df_classified['Any_Match'].isna().sum()
        
        summary.append({
            'City': city,
            'Total_Rows': total,
            'Matched': matched,
            'Unjoined': unjoined,
            'Match_Rate': f"{matched/total*100:.1f}%" if matched != 'N/A' else 'N/A',
            'Join_Rate': f"{(total-unjoined)/total*100:.1f}%"})
        
        print(f"  Input/Output rows: {len(df):,} / {total:,} — {'OK' if len(df) == total else 'MISMATCH'}")
        print(f"  Unjoined rows:     {unjoined:,}")
        print(f"  Matched: {matched:,} / {total:,} rows")
        print(f"  Saved: {output_path}")

    except FileNotFoundError:
        print(f"  ERROR: File not found — {filepath}")
        summary.append({'City': city, 'Total_Rows': 'ERROR', 
                        'Matched': 'ERROR', 'Match_Rate': 'ERROR'})
    except KeyError as e:
        print(f"  ERROR: Column not found — {e}")
        summary.append({'City': city, 'Total_Rows': 'ERROR', 
                        'Matched': 'ERROR', 'Match_Rate': 'ERROR'})


print("\n=== PROCESSING SUMMARY ===")
summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
output_summ = os.path.join(OUTPUT_FOLDER, f"summary_classified_NY.csv")
summary_df.to_csv(output_summ, index=False)